In [1]:
import warnings
warnings.filterwarnings("ignore",category=UserWarning)
import pandas as pd
import pyodbc
import time
from datetime import datetime
SOURCE_START_DATE="2025-01-01"
#CHECK_INTERVAL_SECONDS=60
def create_connection():
    conn=pyodbc.connect(
        "DRIVER={ODBC Driver 18 for SQL Server};"
        "SERVER=atwpSQL-STP-app;"
        "DATABASE=LinePC7442;"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )
    return conn

"""Deletes all records from the stable production result table.
 If the delete operation fails, the transaction is rolled back so
    that the database is not left in an unfinished state."""
def clear_stable_production_table():
    conn=create_connection()
    try:
        cursor=conn.cursor()
        cursor.execute("""
            DELETE FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
        """)
        deleted_rows=cursor.rowcount
        conn.commit()
        cursor.close()
        print(f"Deleted {deleted_rows} rows from Ex2_stableProd_dev.")
    except Exception as e:
        conn.rollback()
        print(f"Error clearing Ex2_stableProd_dev: {e}")
        raise
    finally:
        conn.close()

"""Checks whether the stable production result table contains any rows.
    Instead of loading the whole table, the query only looks for one row.
    This makes the check quick even when the table contains a lot of data."""
# def target_table_is_empty(conn):
#     query="""
#         SELECT TOP 1 1
#         FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
#     """
#     cursor=conn.cursor()
#     cursor.execute(query)
#     result=cursor.fetchone()
#     cursor.close()
#     return result is None

"""Gets the latest timestamp currently stored in the result table.
    There are two possible timestamps used in the result table:
    - stable_ts_stop is used for valid production runs.
    - processtime is used when a combination was checked but no valid
      production run was found.
    The function looks at both and returns the newest timestamp."""
# def get_latest_target_timestamp(conn):
    
#     query="""
#         SELECT MAX(ts)
#         FROM (
#             SELECT MAX([stable_ts_stop]) AS ts
#             FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
#             WHERE [stable_ts_stop] IS NOT NULL
#             UNION ALL
#             SELECT MAX([processtime]) AS ts
#             FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
#             WHERE [stable_ts_stop] IS NULL
#         ) 
#     """
#     cursor=conn.cursor()
#     cursor.execute(query)
#     result=cursor.fetchone()
#     cursor.close()
#     if result is None or result[0] is None:
#         return None
#     return pd.to_datetime(result[0])

"""Checks whether a Prog_Nr and ordername combination already exists
    in the stable production result table.
    This is used to avoid processing the same program/order combination
    more than once."""
def combination_exists(conn,prog_nr,ordername):
    query="""
        SELECT TOP 1 1
        FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
        WHERE [Prog_Nr]=?
          AND [ordername]=?
    """
    cursor=conn.cursor()
    cursor.execute(query,(prog_nr,ordername))
    result=cursor.fetchone()
    cursor.close()
    return result is not None

"""Gets every unique Prog_Nr and ordername combination from the source
    production data.
    For each combination, the latest available source timestamp is also
    returned.

    This function does not check whether the combination has already
    been processed. It simply returns the complete list from the source
    table."""
# def get_program_orders_full(conn):
#     query=f"""
#         SELECT
#             CAST([Prog_Nr] AS VARCHAR(255)) AS Prog_Nr,
#             CAST([ordername] AS VARCHAR(255)) AS ordername,
#             MAX([timestamp]) AS latest_ts
#         FROM [LinePC7442].[dbo].[DI_Ex2_Istwerte]
#         WHERE [timestamp]>='{SOURCE_START_DATE}'
#           AND [Prog_Nr] IS NOT NULL
#           AND [ordername] IS NOT NULL
#         GROUP BY
#             CAST([Prog_Nr] AS VARCHAR(255)),
#             CAST([ordername] AS VARCHAR(255))
#         ORDER BY latest_ts ASC
#     """
#     return pd.read_sql(query,conn)

"""Finds Prog_Nr and ordername combinations that exist in the source
    table but have not yet been added to the stable production result table.
    This is the main function used for incremental processing."""
def get_new_program_orders(conn):
    query=f"""
    WITH Orders AS
    (
        SELECT
            CAST([Prog_Nr] AS VARCHAR(255)) AS Prog_Nr,
            CAST([ordername] AS VARCHAR(255)) AS ordername,
            MAX([timestamp]) AS latest_ts
        FROM [LinePC7442].[dbo].[DI_Ex2_Istwerte]
        WHERE [timestamp] >= '{SOURCE_START_DATE}'
        GROUP BY
            CAST([Prog_Nr] AS VARCHAR(255)),
            CAST([ordername] AS VARCHAR(255))
    )
    SELECT *
    FROM Orders o
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM [LinePC7442].[dbo].[Ex2_stableProd_dev] s
        WHERE s.[Prog_Nr]=o.Prog_Nr
          AND s.[ordername]=o.ordername
    )
    ORDER BY latest_ts ASC
    """
    return pd.read_sql(query,conn)

"""Checks how many valid production runs already exist for a given
    Prog_Nr and ordername.

    It also returns the highest production run number that has already
    been assigned."""
def get_existing_production_runs(conn,prog_nr,ordername):
    query="""
        SELECT
            COUNT(*),
            ISNULL(MAX([productions_runs]),0)
        FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
        WHERE [Prog_Nr]=?
          AND [ordername]=?
          AND [valid]=1
    """
    cursor=conn.cursor()
    cursor.execute(query,(prog_nr,ordername))
    result=cursor.fetchone()
    cursor.close()
    return int(result[0]),int(result[1])

"""Loads the production measurements for one specific program/order
    combination.
    The query also removes data that is already covered by a previously
    detected valid production run. This prevents the same production
    period from being analysed again.
    After loading the data, the function converts timestamps and numeric
    columns into the correct Python/Pandas data types and removes rows
    where the important values are missing."""

def get_production_data(conn,prog_nr,ordername):
    query="""
        SELECT
            d.[timestamp],
            CAST(d.[Prog_Nr] AS VARCHAR(255)) AS [Prog_Nr],
            d.[ordername],
            d.[Extr_ist] AS rpm,
            d.[Auszen_DM_XY_ist] AS OD_real,
            ISNULL(d.[Ausstoz_ist],0) AS Ausstoz_ist,
            d.[Massedruck],
            d.[Mass_ist],
            d.[Linie_ist] AS line_speed
        FROM [LinePC7442].[dbo].[DI_Ex2_Istwerte] d
        WHERE d.[timestamp]>=?
          AND CAST(d.[Prog_Nr] AS VARCHAR(255))=?
          AND d.[ordername]=?
          AND NOT EXISTS
          (
              SELECT 1
              FROM [LinePC7442].[dbo].[Ex2_stableProd_dev] s
              WHERE s.[valid]=1
                AND s.[Prog_Nr]=CAST(d.[Prog_Nr] AS VARCHAR(255))
                AND s.[ordername]=d.[ordername]
                AND d.[timestamp]>=s.[stable_ts_start]
                AND d.[timestamp]<=s.[stable_ts_stop]
          )
        ORDER BY d.[timestamp]
    """
    df=pd.read_sql(query,conn,params=[SOURCE_START_DATE,str(prog_nr),ordername])
    if df.empty:
        return df
    df["timestamp"]=pd.to_datetime(df["timestamp"],errors="coerce")
    numeric_columns=["rpm","OD_real","Ausstoz_ist","Massedruck","Mass_ist","line_speed"]
    for col in numeric_columns:
        df[col]=pd.to_numeric(df[col],errors="coerce")
    df=df.dropna(subset=["timestamp","rpm","OD_real","Ausstoz_ist"])
    return df.sort_values("timestamp").reset_index(drop=True)
def find_production_segments(df):
    if df.empty:
        return []
    df=df.sort_values("timestamp").reset_index(drop=True).copy()
    df["production_m"]=(df["line_speed"]/60.0)*2
    df["valid"]=(
        (df["rpm"]>0)&
        (df["OD_real"]>1.0)&
        (df["Ausstoz_ist"]>0.05)
    )
    segments=[]
    start_index=None
    invalid_start=None
    MAX_GAP=pd.Timedelta(minutes=1)
    for i,row in df.iterrows():
        if row["valid"]:
            if start_index is None:
                start_index=i
            invalid_start=None
        else:
            if start_index is None:
                continue
            if invalid_start is None:
                invalid_start=row["timestamp"]
            gap=row["timestamp"]-invalid_start
            if gap>MAX_GAP:
                end_index=i-1
                while end_index>=start_index and not df.loc[end_index,"valid"]:
                    end_index-=1
                if end_index>=start_index:
                    segments.append(df.loc[start_index:end_index].copy())
                start_index=None
                invalid_start=None
    if start_index is not None:
        end_index=len(df)-1
        while end_index>=start_index and not df.loc[end_index,"valid"]:
            end_index-=1
        if end_index>=start_index:
            segments.append(df.loc[start_index:end_index].copy())
    production_results=[]
    for segment in segments:
        if segment.empty:
            continue
        raw_start=segment.iloc[0]["timestamp"]
        raw_end=segment.iloc[-1]["timestamp"]
        raw_minutes=(raw_end-raw_start).total_seconds()/60.0
        if raw_minutes<15:
            continue
        if len(production_results)==0:
            ignore_time=raw_start+pd.Timedelta(minutes=9)
            after_start=segment[segment["timestamp"]>=ignore_time]
            if after_start.empty:
                continue
            stable_start=after_start.iloc[0]["timestamp"]
        else:
            stable_start=raw_start
        stable_stop=segment.iloc[-1]["timestamp"]
        stable_minutes=(stable_stop-stable_start).total_seconds()/60.0
        if stable_minutes<15:
            continue
        stable_data=segment[
            (segment["timestamp"]>=stable_start) &
            (segment["timestamp"]<=stable_stop)
        ].copy()

        stable_meters=stable_data["production_m"].sum()
        production_results.append({
            "ramp_up":stable_start,
            "ramp_down":stable_stop,
            "stable_minutes":stable_minutes,
            "stable_meters":stable_meters
        })
    return production_results

"""Finds the next production run number for a specific program/order.
    For example, if runs 1, 2 and 3 already exist, this function
    returns 4."""
def get_next_run_number(conn,prog_nr,ordername):
    query="""
        SELECT ISNULL(MAX([productions_runs]),0)+1
        FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
        WHERE [Prog_Nr]=?
          AND [ordername]=?
          AND [valid]=1
    """
    cursor=conn.cursor()
    cursor.execute(query,(prog_nr,ordername))
    next_number=cursor.fetchone()[0]
    cursor.close()
    return int(next_number)

"""Saves one detected stable production run into the result table.

    The function stores the program number, order name, start and stop
    timestamps, production run number and the calculated production
    duration.

    The result is marked as valid by setting valid = 1.

    Some additional columns are currently inserted as NULL because
    they are not calculated yet in this part of the program."""
def insert_production_run(conn,prog_nr,ordername,stable_start,stable_stop,production_run,prod_run_time,stable_meters):
    query="""
        INSERT INTO [LinePC7442].[dbo].[Ex2_stableProd_dev]
        (
            [Prog_Nr],
            [ordername],
            [stable_ts_start],
            [stable_ts_stop],
            [productions_runs],
            [valid],
            [processtime],
            [prodRun_time],
            [description],
            [line_speed_min],
            [line_speed_max],
            [line_speed_avg],
            [line_speed_std],
            [stable_meters]
        )
        VALUES
        (?,?,?,?,?,1,CURRENT_TIMESTAMP,?,NULL,NULL,NULL,NULL,NULL,?)
    """
    cursor=conn.cursor()
    cursor.execute(query,
        (
            prog_nr,
            ordername,
            stable_start,
            stable_stop,
            int(production_run),
            float(prod_run_time),
            float(stable_meters)
        )
    )
    cursor.close()
"""
    Saves a program/order combination as an invalid result when no
    valid production run could be found.

    Before inserting the record, the function checks whether the
    combination already exists. This prevents duplicate invalid
    records.

    The record is marked with valid = 0 and a production run number
    of 0. """
def insert_invalid_result(conn,prog_nr,ordername):
    query="""
        IF NOT EXISTS
        (
            SELECT 1
            FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
            WHERE [Prog_Nr]=?
              AND [ordername]=?
        )
        BEGIN
            INSERT INTO [LinePC7442].[dbo].[Ex2_stableProd_dev]
            (
                [Prog_Nr],
                [ordername],
                [stable_ts_start],
                [stable_ts_stop],
                [productions_runs],
                [valid],
                [processtime],
                [prodRun_time],
                [description],
                [line_speed_min],
                [line_speed_max],
                [line_speed_avg],
                [line_speed_std],
                [stable_meters]
            )
            VALUES
            (?,?,NULL,NULL,0,0,CURRENT_TIMESTAMP,0,NULL,NULL,NULL,NULL,NULL,NULL)
        END
    """
    cursor=conn.cursor()
    cursor.execute(query,(prog_nr,ordername,prog_nr,ordername))
    cursor.close()

    """Processes one Prog_Nr and ordername combination from start to finish.

    The function first checks whether this combination has already been
    processed. If it has, nothing else is done."""
def process_program_order(conn,prog_nr,ordername):
    print()
    print(f"Processing Prog_Nr={prog_nr} | Order={ordername}")
    # existing_count,existing_max_run=get_existing_production_runs(conn,prog_nr,ordername)
    # if existing_count>0:
    #     print(f"Existing stable production runs found: {existing_count}")
    #     print(f"Last production run number: {existing_max_run}")
    #     print("Already processed.")
    #     return 0,True
    if combination_exists(conn,prog_nr,ordername):
        print("Already processed.")
        return 0,True
    existing_count,existing_max_run=get_existing_production_runs(conn,prog_nr,ordername)
    if existing_count>0:
        print(f"Existing stable production runs found: {existing_count}")
        print(f"Last production run number: {existing_max_run}")
        print("Already processed.")
        return 0,True
    print("No existing stable production runs found.")
    df=get_production_data(conn,prog_nr,ordername)
    if df.empty:
        print("No new production data found.")
        print("No production data is available for this order.")
        insert_invalid_result(conn,prog_nr,ordername)
        conn.commit()
        return 0,False
    print(f"Production data rows found: {len(df)}")
    production_segments=find_production_segments(df)
    if not production_segments:
        print("No valid production runs found.")
        insert_invalid_result(conn,prog_nr,ordername)
        conn.commit()
        return 0,False
    next_run=get_next_run_number(conn,prog_nr,ordername)
    inserted_count=0
    for production in production_segments:
        stable_start=production["ramp_up"]
        stable_stop=production["ramp_down"]
        stable_minutes=production["stable_minutes"]
        stable_meters=production["stable_meters"]
        print()
        print(f"Production Run {next_run}")
        print(f"Stable Start : {stable_start}")
        print(f"Stable Stop  : {stable_stop}")
        print(f"Duration     : {stable_minutes:.2f} minutes")
        print(f"Stable Meters: {stable_meters:.2f} m")
        insert_production_run(conn,prog_nr,ordername,stable_start,stable_stop,next_run,stable_minutes,stable_meters)
        inserted_count+=1
        next_run+=1
    conn.commit()
    print()
    print(f"Inserted production runs: {inserted_count}")
    return inserted_count,False

"""Runs the incremental production analysis.

    The function connects to the database and looks for new
    Prog_Nr + ordername combinations that have not been processed yet.

    Each new combination is then sent to process_program_order(),
    which analyses its production data and saves the result.

    At the end, a summary is printed showing how many orders were
    processed and how many production runs were inserted.

    If something goes wrong, the database transaction is rolled back
    and the error is raised again so that it can be handled by the
    calling program.
    """
def run_latest_analysis():
    conn=create_connection()
    try:
        program_orders=get_new_program_orders(conn)
        if program_orders.empty:
            print()
            print("No new Prog_Nr + Order combinations found.")
            print(f"Checked at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            return
        print()
        print(f"Found {len(program_orders)} new Prog_Nr + Order combinations.")
        total_inserted=0
        processed_orders=0
        for _,row in program_orders.iterrows():
            prog_nr=str(row["Prog_Nr"])
            ordername=str(row["ordername"])
            print()
            print("="*70)
            print(f"New order found:")
            print(f"Prog_Nr={prog_nr}")
            print(f"Order={ordername}")
            print(f"Source latest timestamp={row['latest_ts']}")
            print("="*70)
            inserted_count,_=process_program_order(conn,prog_nr,ordername)
            total_inserted+=inserted_count
            processed_orders+=1
        print()
        print("="*70)
        print("Incremental production analysis completed.")
        print(f"New orders processed: {processed_orders}")
        print(f"Total runs inserted: {total_inserted}")
        print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*70)
    except Exception as e:
        conn.rollback()
        print()
        print(f"ERROR during analysis: {e}")
        raise
    finally:
        conn.close()
# def run_automated_analysis():
#     print("="*70)
#     print("AUTOMATED STABLE PRODUCTION ANALYSIS")
#     #print(f"Check interval: {CHECK_INTERVAL_SECONDS} seconds")
#     print("="*70)
#     while True:
#         try:
#             print()
#             print("="*70)
#             print(f"Starting analysis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
#             print("="*70)
#             run_latest_analysis()
#             print()
#             #print(f"Waiting {CHECK_INTERVAL_SECONDS} seconds for new orders...")
#             #time.sleep(CHECK_INTERVAL_SECONDS)
#         except KeyboardInterrupt:
#             print()
#             print("Automation stopped by user.")
#             break
#         except Exception as e:
#             print()
#             print(f"Automation error: {e}")
#             #print(f"Retrying in {CHECK_INTERVAL_SECONDS} seconds...")
#             #time.sleep(CHECK_INTERVAL_SECONDS)
if __name__=="__main__":
    #clear_stable_production_table()
    #run_automated_analysis()
    run_latest_analysis()



Found 3 new Prog_Nr + Order combinations.

New order found:
Prog_Nr=2927
Order=6342
Source latest timestamp=2026-09-21 07:26:13

Processing Prog_Nr=2927 | Order=6342
No existing stable production runs found.
Production data rows found: 1536

Production Run 1
Stable Start : 2026-09-21 06:44:31
Stable Stop  : 2026-09-21 07:16:45
Duration     : 32.23 minutes
Stable Meters: 163.76 m

Inserted production runs: 1

New order found:
Prog_Nr=2114
Order=6346
Source latest timestamp=2026-09-21 08:51:15

Processing Prog_Nr=2114 | Order=6346
No existing stable production runs found.
Production data rows found: 1357

Production Run 1
Stable Start : 2026-09-21 08:20:13
Stable Stop  : 2026-09-21 08:51:01
Duration     : 30.80 minutes
Stable Meters: 164.36 m

Inserted production runs: 1

New order found:
Prog_Nr=2130
Order=6344
Source latest timestamp=2026-09-21 11:04:17

Processing Prog_Nr=2130 | Order=6344
No existing stable production runs found.
Production data rows found: 592
No valid production r